In [1]:
import spark_config
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("LogAnalyzer") \
    .master("local[*]") \
    .config("spark.driver.memory", "4g") \
    .getOrCreate()

print(f"Spark {spark.version} ready!")

Spark 3.5.7 ready!


In [2]:
import os

BASE_DIR = os.path.dirname(os.getcwd()) if "notebooks" in os.getcwd() else os.getcwd()
RAW_DATA_DIR = os.path.join(BASE_DIR, "data", "raw")

csv_files = [f for f in os.listdir(RAW_DATA_DIR) if f.endswith(".csv")]
print(f"Found: {csv_files}")

file_path = os.path.join(RAW_DATA_DIR, csv_files[0])
df = spark.read.csv(file_path, header=True, inferSchema=True)

print(f"Rows: {df.count():,}")
print(f"Columns: {df.columns}")
df.show(5)

Found: ['labeled.csv']
Rows: 9,282,184
Columns: ['ip', 'time', 'method', 'url', 'protocol', 'status', 'size', 'referrer', 'user_agent', 'extra', 'no', 'label', 'type']
+-------------+-------------------+------+--------------------+--------+------+-----+--------------------+--------------------+-----+---+-----+------+
|           ip|               time|method|                 url|protocol|status| size|            referrer|          user_agent|extra| no|label|  type|
+-------------+-------------------+------+--------------------+--------+------+-----+--------------------+--------------------+-----+---+-----+------+
|  31.56.96.51|2019-01-22 05:56:16|   GET|/image/60844/prod...|HTTP/1.1|   200| 5667|https://www.zanbi...|Mozilla/5.0 (Linu...|    -|  2|    0|benign|
|  31.56.96.51|2019-01-22 05:56:16|   GET|/image/61474/prod...|HTTP/1.1|   200| 5379|https://www.zanbi...|Mozilla/5.0 (Linu...|    -|  3|    0|benign|
|  91.99.72.15|2019-01-22 05:56:17|   GET|/product/31893/62...|HTTP/1.1|   20

In [3]:
print(f"Number of partitions: {df.rdd.getNumPartitions()}")

Number of partitions: 22


In [4]:
from pyspark.sql.functions import col

# Valid values we identified from our exploration
valid_protocols = ["HTTP/1.1", "HTTP/1.0"]
valid_types = ["benign", "bot", "sqli", "scanning", "rce"]
valid_labels = ["0", "1"]

# Keep only rows where these columns have valid values
df_clean = df.filter(
    col("protocol").isin(valid_protocols) &
    col("type").isin(valid_types) &
    col("label").isin(valid_labels)
)

original_count = df.count()
clean_count = df_clean.count()
dropped = original_count - clean_count

print(f"Original rows:  {original_count:,}")
print(f"Clean rows:     {clean_count:,}")
print(f"Dropped rows:   {dropped:,} ({(dropped/original_count)*100:.2f}%)")

Original rows:  9,282,184
Clean rows:     9,245,740
Dropped rows:   36,444 (0.39%)


In [5]:
from pyspark.sql.functions import col
from pyspark.sql.types import IntegerType, LongType

# Drop the 'extra' column (99.35% empty, useless)
# Cast columns to proper types
df_clean = df_clean \
    .drop("extra") \
    .withColumn("label", col("label").cast(IntegerType())) \
    .withColumn("no", col("no").cast(LongType())) \
    .withColumn("size", col("size").cast(LongType())) \
    .withColumn("status", col("status").cast(IntegerType()))

print("Updated schema:")
df_clean.printSchema()
df_clean.show(5)

Updated schema:
root
 |-- ip: string (nullable = true)
 |-- time: timestamp (nullable = true)
 |-- method: string (nullable = true)
 |-- url: string (nullable = true)
 |-- protocol: string (nullable = true)
 |-- status: integer (nullable = true)
 |-- size: long (nullable = true)
 |-- referrer: string (nullable = true)
 |-- user_agent: string (nullable = true)
 |-- no: long (nullable = true)
 |-- label: integer (nullable = true)
 |-- type: string (nullable = true)

+-------------+-------------------+------+--------------------+--------+------+-----+--------------------+--------------------+---+-----+------+
|           ip|               time|method|                 url|protocol|status| size|            referrer|          user_agent| no|label|  type|
+-------------+-------------------+------+--------------------+--------+------+-----+--------------------+--------------------+---+-----+------+
|  31.56.96.51|2019-01-22 05:56:16|   GET|/image/60844/prod...|HTTP/1.1|   200| 5667|https://www

In [6]:
import os

CLEAN_DATA_DIR = os.path.join(BASE_DIR, "data", "clean")
os.makedirs(CLEAN_DATA_DIR, exist_ok=True)

# Save as parquet (much faster and smaller than CSV for Spark)
df_clean.write.mode("overwrite").parquet(os.path.join(CLEAN_DATA_DIR, "logs_clean.parquet"))

print(f"Saved clean data to: {CLEAN_DATA_DIR}")
print(f"Clean rows: {df_clean.count():,}")

Saved clean data to: C:\Users\amarbir\Self_Learning\spark_flask_log_app\data\clean
Clean rows: 9,245,740
